In [ ]:
#| default_exp machine_learning.note_linking

In [ ]:
#| export

from itertools import combinations
import random
from typing import TypedDict

from jarowinkler import jarowinkler_similarity 



from trouver.machine_learning.note_data import (
    NoteLinkEnum, NoteData, InfoNoteData, NotatNoteData, _update_dict)



In [ ]:
from unittest.mock import MagicMock

## Sieve instances of pairs for building a dataset

**Data Sieving and Negative Sampling**

A convenient feature of `Obsidian.md` is that notes can have linkts and embedded links to other notes. For mathematical text, such links help to remind oneself, for example, of the meaning of various definitions and notations that a particular note might depend on. We attempt to train and use machine learning [ML] models for link prediction, which should largely consist of two prediction tasks:
1. Understand if one note should link to another (and what the general "rationale" for doing so is)
2. Find where in the note a link should best be positioned.  

When it comes to training a machine learning model for link prediction, creating a dataset is an asymmetry problem:

1. Positive Instances are easy: If Note A links to Note B, that is a positive sample.
2. Negative Instances are hard: If Note A does not link to Note B, does that mean they shouldn't link, or simply that the author hasn't added the link yet?

To solve this, we use a Heuristic Sieving Strategy:
1. The "Well-Connected" Heuristic

We assume that notes with many existing links (both incoming and outgoing) are "mature" or "well-focused." If a mature note does not link to another note, it is high-probability evidence of a true negative.

- High Count Notes: High incoming links ($>4$) and valid outgoing links ($>2$). These are the "anchors" of our dataset.
- Mid Count Notes: Moderate incoming links ($>2$) and outgoing links ($>1$).

2. Hard Negative Mining (Similarity)

Models often struggle to distinguish between distinct concepts that share similar names or notation (e.g., distinguishing $\mathcal{F}$ the sheaf from $F$ the field).

- Similar Notation Injection: We calculate the Jaro-Winkler distance between notation strings.

- We deliberately inject pairs of notes with similar notations but no actual link into the dataset. This forces the model to learn context beyond just surface-level string similarity.

3. Admissibility Filtering

Finally, we filter pairs based on _auto tags. If a note has already been processed by an automated linker (_auto/links_added), we respect its current state to avoid retraining on model-generated noise.

<!-- In practice, it is difficult to manually create all links that ought to be linked. In particular, while it can be easy to extract "positive" instances of links (by virtue of simply finding explicit links), it is more difficult to obtain "negative" instances of links with certainty. The general method for obtaining "negative" instances is nevertheless to randomly sample pairs of notes and consider such a pair as "negative" if there is no link between them; some notes are "well focused" on in practice (and in particular has many links to other notes); if there are no links between two such notes, then it is likely that there are not supposed to be links between them. -->

In [ ]:
#| export
class NotePairData(TypedDict):
    origin_note: NoteData
    relied_note: NoteData
    # linked_type: NoteLinkEnum

In [ ]:
#| export
def link_types_for_note_pair_data(
        pair_data: NotePairData
        ) -> set[NoteLinkEnum]:
    relied_note_name: str = pair_data['relied_note'].note_name
    directly_linked_notes_from_origin = pair_data['origin_note'].directly_linked_notes
    if relied_note_name in directly_linked_notes_from_origin:
        return set(directly_linked_notes_from_origin[relied_note_name])
    else:
        return set([NoteLinkEnum.NO_LINK])

In [ ]:
#| export
def _high_count_note_data(
        info_note_data: dict[str, InfoNoteData],
        notat_note_data: dict[str, NotatNoteData],
        ) -> tuple[set[str], set[str]]:
    """
    Helper function to `sieve_note_data_pairs`.
    """

    high_count_info_notes: set[str] = set([
        name for name, data_point in info_note_data.items()
        if len(data_point.reverse_linked_notes) > 4
        and len(info_note_data[name].directly_linked_notes) > 2])
    high_count_notat_notes: set[str] = set([
        name for name, data_point in notat_note_data.items()
        if len(data_point.reverse_linked_notes) > 4])

    return (high_count_info_notes, high_count_notat_notes)


def _mid_count_note_data(
        info_note_data: dict[str, InfoNoteData],
        notat_note_data: dict[str, NotatNoteData],
        high_count_info_notes: set[str],
        high_count_notat_notes: set[str],
        ) -> tuple[set[str], set[str]]:
    """
    Helper function to `sieve_note_data_pairs`.
    """

    mid_count_info_notes: set[str] = set([
        name for name, data_point in info_note_data.items()
        if len(data_point.reverse_linked_notes) > 2
        and len(info_note_data[name].directly_linked_notes) > 1
        and name not in high_count_info_notes])
    mid_count_notat_notes: list[str] = set([
        name for name, data_point in notat_note_data.items()
        if len(data_point.reverse_linked_notes) > 2
        and name not in high_count_notat_notes])

    return (mid_count_info_notes, mid_count_notat_notes)

In [ ]:
#| export
def _positive_instances_from_high_or_mid_count_notes(
        high_count_notes: set[str],
        mid_count_notes: set[str],
        note_data: dict[str, NoteData],
        ) -> list[tuple[str, str]]:
    """
    Helper function to `sieve_note_data_pairs`.
    """
    chosen_pairs: list[tuple[str, str]] = []
    # Get all "positive" note links from high count notes to high or mid count notes.
    for high_count_note_name in list(high_count_notes):
    # for high_count_note_name, high_count_data_point in high_count_notes.items():
        high_count_data_point = note_data[high_count_note_name]
        for other_note, _ in high_count_data_point.directly_linked_notes.items():
            if other_note in high_count_notes or other_note in mid_count_notes:
                chosen_pairs.append((high_count_note_name, other_note))
    # Get all "positive" note links from mid count notes to high count notes.
    for mid_count_note_name in list(mid_count_notes):
        mid_count_data_point = note_data[mid_count_note_name]
    # for mid_count_note_name, mid_count_data_point in mid_count_notes.items():
        for other_note, _ in mid_count_data_point.directly_linked_notes.items():
            if other_note in high_count_notes:
                chosen_pairs.append((mid_count_note_name, other_note))
    return chosen_pairs

In [ ]:
#| export
def _negative_instances_from_high_or_mid_count_notes(
        high_count_notes: set[str],
        mid_count_notes: set[str],
        note_data: dict[str, NoteData],
        num_pairs: int # The approximate number of pairs to sample.
        ) -> list[tuple[str, str]]:
    """
    Get "negative" pair instances from high or mid count notes, i.e. pairs where
    the origin note seem to not link to relied note.
    """
    high_count_weights = [
        (len(note_data[note_name].reverse_linked_notes)**0.5)
        for note_name in list(high_count_notes)]
    mid_count_weights = [
        (len(note_data[note_name].reverse_linked_notes)**0.5)
        for note_name in list(mid_count_notes)]
    high_to_high_samples = int(0.5 * num_pairs)
    high_to_mid_samples = int(0.25 * num_pairs)
    mid_to_high_samples = int(0.25 * num_pairs)
    high_count_notes_list = list(high_count_notes)
    mid_count_notes_list = list(mid_count_notes)

    sample_pairs: set[tuple[str, str]] = set()

    origin_notes = random.choices(
        high_count_notes_list, weights=high_count_weights, k=high_to_high_samples
        ) if high_count_notes_list else []
    relied_notes = random.choices(
        high_count_notes_list, weights=high_count_weights, k=high_to_high_samples
        ) if high_count_notes_list else []
    for origin_note, relied_note in zip(origin_notes, relied_notes):
        if (origin_note == relied_note
                or relied_note in note_data[origin_note].directly_linked_notes):
            continue
        else:
            sample_pairs.add((origin_note, relied_note))

    origin_notes = random.choices(
        high_count_notes_list, weights=high_count_weights, k=high_to_mid_samples
        ) if high_count_notes_list else []
    relied_notes = random.choices(
        mid_count_notes_list, weights=mid_count_weights, k=high_to_mid_samples
        ) if mid_count_notes_list else []
    for origin_note, relied_note in zip(origin_notes, relied_notes):
        if (origin_note == relied_note
                or relied_note in note_data[origin_note].directly_linked_notes):
            continue
        else:
            sample_pairs.add((origin_note, relied_note))

    origin_notes = random.choices(
        mid_count_notes_list, weights=mid_count_weights, k=mid_to_high_samples
        ) if mid_count_notes_list else []
    relied_notes = random.choices(
        high_count_notes_list, weights=high_count_weights, k=mid_to_high_samples
        ) if high_count_notes_list else []
    for origin_note, relied_note in zip(origin_notes, relied_notes):
        if (origin_note == relied_note
                or relied_note in note_data[origin_note].directly_linked_notes):
            continue
        else:
            sample_pairs.add((origin_note, relied_note))

    return list(sample_pairs)

In [ ]:
#| export
def _similar_notation_pairs(
        notat_note_data: dict[str, NotatNoteData],
        # ) -> list[tuple[str, str]]:
        ) -> dict[str, set[str]]: # The keys are names of notation notes and the values are sets of names of notation notes whose notations are similar to the one explained in the key notation note.
    """
    Identify pairs of names of notation notes whose notations are similar.

    Helper function to sieve_note_data_pairs.

    The similarity is measured by Jaro-Winkler, which works well on short
    strings.
    """
    # jarowinkler = JaroWinkler()
    # similar_notation_pairs: list[tuple[str, str]] = []
    similar_notation_dict: dict[str, set[str]] = {}
    for notat_name_1, notat_name_2 in combinations(notat_note_data, 2):
        notat_data_1, notat_data_2 = notat_note_data[notat_name_1], notat_note_data[notat_name_2]
        notat_str_1 = notat_data_1.parsed.notation_str
        notat_str_2 = notat_data_2.parsed.notation_str
        similarity = jarowinkler_similarity(notat_str_1, notat_str_2)
        reverse_similarity = jarowinkler_similarity(notat_str_1[::-1], notat_str_2[::-1]) 
        if similarity > 0.9 or reverse_similarity > 0.9:
            _update_dict(similar_notation_dict, notat_name_1, notat_name_2)
            _update_dict(similar_notation_dict, notat_name_2, notat_name_1)
    return similar_notation_dict

In [ ]:
#| export
def _random_pair_replacing_notation_notes_with_similar_notation_notes(
        original_pair: tuple[str, str],
        similar_notation_dict: set[str, set[str]]
        ) -> tuple[str, str]:
    """
    Helper function to `_random_pair_replacing_notation_notes_with_similar_notation_notes`.
    """
    origin_note_name = original_pair[0]
    relied_note_name = original_pair[1]
    if random.random() > 0.5:
        if origin_note_name in similar_notation_dict:
            origin_note_name = random.choice(list(similar_notation_dict[origin_note_name]))
    if random.random() > 0.5:
        if relied_note_name in similar_notation_dict:
            relied_note_name = random.choice(list(similar_notation_dict[relied_note_name]))
    return (origin_note_name, relied_note_name)
    

    
def _pairs_with_notation_notes_replaced_with_similar_notation_notes(
        sieved_pairs: set[tuple[str, str]],
        count: int, # The approximate number of pairs to attempt to obtain.
        similar_notation_dict: set[str, set[str]], # An output of `_similar_notation_pairs`
        # notat_note_data: dict[str, NotatNoteData],
        ) -> list[tuple[str, str]]:
    """
    Return modified versions of entries of `sieved_pairs` drawn at random
    where notation note names are replaced by names of notation notes whose 
    introduced notations are similar, in accordance to `similar_notation_dict`.

    Helper function to `sieve_note_data_pairs`.
    """
    sieved_pairs_list = list(sieved_pairs)
    new_pairs: list[tuple[str, str]] = []
    for _ in range(count):
        original_pair = random.choice(sieved_pairs_list)
        new_pair = _random_pair_replacing_notation_notes_with_similar_notation_notes(
            original_pair, similar_notation_dict)
        new_pairs.append(new_pair)
    return new_pairs

In [ ]:
#| export
def _pair_is_admissible(
        origin_note: str,
        relied_note: str,
        note_data: dict[str, NoteData],
        info_note_data: dict[str, InfoNoteData],
        notat_note_data: dict[str, InfoNoteData],
        ) -> bool:
    origin_note_has_tags = note_data[origin_note].tags is not None
    if not origin_note_has_tags:
        return True
    if (('_auto/links_added' in note_data[origin_note].tags and relied_note in info_note_data)
            or ('_auto/notations_added' in note_data[origin_note].tags and relied_note in notat_note_data)):
        return False
    return True

In [ ]:

#| export
def _classify_anchor_notes(
        info_note_data: dict[str, InfoNoteData],
        notat_note_data: dict[str, NotatNoteData]
        ) -> tuple[set[str], set[str], set[str], set[str]]:
    """Hidden helper: Classify notes into High/Mid counts for sieving."""
    high_info, high_notat = _high_count_note_data(info_note_data, notat_note_data)
    mid_info, mid_notat = _mid_count_note_data(info_note_data, notat_note_data, high_info, high_notat)
    return high_info | high_notat, mid_info | mid_notat, high_info, high_notat

def _gather_raw_pairs(
        high_notes: set[str],
        mid_notes: set[str],
        note_data: dict[str, NoteData],
        notat_note_data: dict[str, NotatNoteData]
        ) -> set[tuple[str, str]]:
    """Hidden helper: Collect positive, negative, and similar-notation pairs."""
    pos_pairs = _positive_instances_from_high_or_mid_count_notes(high_notes, mid_notes, note_data)
    neg_pairs = _negative_instances_from_high_or_mid_count_notes(
        high_notes, mid_notes, note_data, num_pairs=len(pos_pairs)*3)
    
    sieved_pairs = set(pos_pairs) | set(neg_pairs)
    
    # Add hard negatives (similar notation)
    sim_dict = _similar_notation_pairs(notat_note_data)
    sim_pairs = _pairs_with_notation_notes_replaced_with_similar_notation_notes(
        sieved_pairs, len(pos_pairs), sim_dict)
    
    return sieved_pairs | set(sim_pairs)

def _filter_admissible_pairs(
        candidate_pairs: set[tuple[str, str]],
        note_data: dict[str, NoteData],
        info_data: dict[str, InfoNoteData],
        notat_data: dict[str, NotatNoteData]
        ) -> list[NotePairData]:
    """Hidden helper: Convert valid raw pairs into NotePairData objects."""
    final_data = []
    for origin, relied in candidate_pairs:
        if _pair_is_admissible(origin, relied, note_data, info_data, notat_data):
            final_data.append(NotePairData(
                origin_note=note_data[origin], 
                relied_note=note_data[relied]))
    return final_data

In [ ]:
#| export
def sieve_note_data_pairs(
        info_note_data: dict[str, InfoNoteData], # Data for all standard information notes.
        notat_note_data: dict[str, NotatNoteData] # Data for all notation notes.
        ) -> list[NotePairData]: # A balanced list of note pairs for training.
    """
    Constructs a balanced training dataset of note pairs by sampling positive links, 
    inferred negative links, and 'hard negative' pairs with similar notation.
    """
    note_data = {**info_note_data, **notat_note_data}
    
    # 1. Identify "Anchor" notes (well-connected notes suitable for sampling)
    high_notes, mid_notes, _, _ = _classify_anchor_notes(info_note_data, notat_note_data)
    
    # 2. Gather raw candidate pairs (Positives + Negatives + Similar Notation)
    raw_pairs = _gather_raw_pairs(high_notes, mid_notes, note_data, notat_note_data)
    
    # 3. Filter for admissibility and format
    return _filter_admissible_pairs(raw_pairs, note_data, info_note_data, notat_note_data)

Sieving Note Pairs for Training Data

The sieve_note_data_pairs function is the core pipeline for assembling a balanced dataset of note pairs. It solves the problem of "implicit negatives" by intelligently sampling pairs that likely shouldn't be linked.

The Sieving Process:

    1. Identify "Anchor" Notes: It classifies notes into "High Count" (well-connected) and "Mid Count" categories based on their link density. We assume these notes are mature enough that missing links are true negatives.

    2. Gather Positives: It collects all existing valid links between these anchor notes.

    3. Sample Negatives: It randomly samples pairs of anchor notes that are not currently linked. To balance classes, it samples ~3x as many negatives as positives.

    4. Inject Hard Negatives: It adds pairs of notes with visually similar notation (e.g., Gal(L/K) vs Gal(F/E)) but no actual link. This forces the model to learn context, not just string similarity.

    5. Filter Admissibility: Finally, it removes pairs where the origin note has already been auto-processed (_auto/links_added), preventing the model from training on its own prior predictions.


In [ ]:
#| example

# 1. Setup Helper (Same as before, but ensure it sets necessary attributes)
def mock_note_data(name, reverse_count=0, direct_links=None, tags=None, notation_str=""):
    m = MagicMock()
    m.note_name = name
    # Satisfy count thresholds with dummy sets
    m.reverse_linked_notes = {f"in_{i}" for i in range(reverse_count)}
    m.directly_linked_notes = direct_links if direct_links else {}
    m.tags = tags
    
    # Setup Notation Data specifics
    m.parsed = MagicMock()
    m.parsed.notation_str = notation_str if notation_str else name
    return m

# 2. Create a "Mature" Dataset that satisfies Sieve Thresholds
#
# Thresholds Reminder:
# - High Count Info:  >4 Reverse Links AND >2 Direct Links
# - High Count Notat: >4 Reverse Links
# - Mid Count Info:   >2 Reverse Links AND >1 Direct Link

info_mock = {
    # High Count Info Note (Anchor)
    # 5 incoming, 3 outgoing -> Qualifies as High Count
    "Info_High": mock_note_data(
        "Info_High", 
        reverse_count=5, 
        direct_links={"Notat_A": 1, "Notat_B": 1, "Info_Mid": 1}
    ),
    
    # Mid Count Info Note
    # 3 incoming, 2 outgoing -> Qualifies as Mid Count
    "Info_Mid": mock_note_data(
        "Info_Mid", 
        reverse_count=3, 
        direct_links={"Info_High": 1, "Notat_A": 1}
    )
}

notat_mock = {
    # High Count Notation Note
    # 5 incoming -> Qualifies as High Count
    "Notat_A": mock_note_data(
        "Notat_A", 
        reverse_count=5, 
        notation_str="Gal(L/K)"
    ),
    
    # Another Notation Note (For Negative Sampling)
    # 5 incoming -> Qualifies as High Count
    "Notat_B": mock_note_data(
        "Notat_B", 
        reverse_count=5, 
        notation_str="Gal(F/E)" # Similar string to Notat_A
    )
}

# 3. Run the Sieve
# This should now find:
# - Positive pairs (e.g., Info_High -> Notat_A)
# - Negative pairs (e.g., Notat_A -> Notat_B, since no link exists)
# - Similar pairs (Notat_A <-> Notat_B due to string similarity)
training_pairs = sieve_note_data_pairs(info_mock, notat_mock)

# 4. Inspect Results
print(f"Generated {len(training_pairs)} training pairs.")
print("-" * 30)

# Group by type for clarity
positives = []
negatives = []

for pair in training_pairs:
    orig = pair['origin_note'].note_name
    dest = pair['relied_note'].note_name
    
    # check if it was an existing link in our mock data
    orig_data = info_mock.get(orig) or notat_mock.get(orig)
    if dest in orig_data.directly_linked_notes:
        positives.append(f"{orig} -> {dest}")
    else:
        negatives.append(f"{orig} -> {dest}")

print("Positive Instances (Existing Links):")
for p in positives[:3]: print(f"  {p}")

print("\nNegative/Inferred Instances (No Link):")
for n in negatives[:3]: print(f"  {n}")

Generated 10 training pairs.
------------------------------
Positive Instances (Existing Links):
  Info_High -> Info_Mid
  Info_High -> Notat_B
  Info_Mid -> Info_High

Negative/Inferred Instances (No Link):
  Notat_A -> Info_High
  Notat_B -> Notat_A
  Notat_B -> Info_Mid


In [ ]:
#| hide
from unittest.mock import MagicMock, patch
from fastcore.test import *

# ---------------------------------------------------------
# Test 1: Link Type Extraction
# ---------------------------------------------------------
def test_link_types_extraction():
    # Adjusted ENUM name to match your file's likely convention (no underscores)
    origin = mock_note_data("Origin", direct_links={"Target": {NoteLinkEnum.INFO_TO_INFO_IN_CONTENT}})
    target = mock_note_data("Target")
    
    pair_data = {"origin_note": origin, "relied_note": target}
    
    # Case 1: Link exists
    types = link_types_for_note_pair_data(pair_data)
    test_eq(types, {NoteLinkEnum.INFO_TO_INFO_IN_CONTENT})
    
    # Case 2: No link exists
    pair_data['relied_note'].note_name = "NonExistent"
    types = link_types_for_note_pair_data(pair_data)
    test_eq(types, {NoteLinkEnum.NO_LINK})

# ---------------------------------------------------------
# Test 2: High/Mid Count Classification
# ---------------------------------------------------------
def test_note_counting_logic():
    # Setup data
    info_data = {
        "A": mock_note_data("A", reverse_count=5, direct_links={"x":1, "y":2, "z":3}),
        "B": mock_note_data("B", reverse_count=3, direct_links={"x":1, "y":2}),
        "C": mock_note_data("C", reverse_count=1)
    }
    notat_data = {} 

    # Test High Count
    high_info, _ = _high_count_note_data(info_data, notat_data)
    test_eq(high_info, {"A"})
    
    # Test Mid Count (Must exclude High counts)
    mid_info, _ = _mid_count_note_data(info_data, notat_data, high_info, set())
    test_eq(mid_info, {"B"})

# ---------------------------------------------------------
# Test 3: Positive Instance Extraction
# ---------------------------------------------------------
def test_positive_instance_extraction():
    notes = {
        "A": mock_note_data("A", direct_links={"B": 1, "C": 1}),
        "B": mock_note_data("B", direct_links={"A": 1}),
        "C": mock_note_data("C")
    }
    
    high_set = {"A"}
    mid_set = {"B"}
    
    pairs = _positive_instances_from_high_or_mid_count_notes(high_set, mid_set, notes)
    pairs.sort() 
    expected = [("A", "B"), ("B", "A")]
    test_eq(pairs, expected)

# ---------------------------------------------------------
# Test 4: Negative Sampling (with Mocked Randomness)
# ---------------------------------------------------------
@patch('random.choices')
def test_negative_instance_sampling(mock_choices):
    notes = {
        "A": mock_note_data("A", reverse_count=5, direct_links={"B": 1}),
        "B": mock_note_data("B", reverse_count=5, direct_links={}) 
    }
    
    # Mock random.choices to return specific sequences:
    # 1. High->High: [B], [A]. Link B->A does NOT exist. (Should be ADDED)
    # 2. High->Mid: Returns empty (no mids in this mock)
    # 3. Mid->High: Returns empty
    
    # Note: Logic inside function calls random.choices 3 times (High->High, High->Mid, Mid->High)
    mock_choices.side_effect = [
        ["B"], ["A"], # High -> High call (origin list, relied list)
        [], [],       # High -> Mid call
        [], []        # Mid -> High call
    ]
    
    high_set = {"A", "B"}
    mid_set = set()
    
    results = _negative_instances_from_high_or_mid_count_notes(
        high_set, mid_set, notes, num_pairs=10
    )
    
    test_eq(results, [("B", "A")])

# ---------------------------------------------------------
# Test 5: Admissibility (Tag Filtering)
# ---------------------------------------------------------
def test_pair_admissibility():
    info_data = {"InfoNote": mock_note_data("InfoNote")}
    notat_data = {"NotatNote": mock_note_data("NotatNote")}
    
    all_data = {**info_data, **notat_data}
    
    # Case 1: Origin has no tags -> Admissible
    all_data["CleanOrigin"] = mock_note_data("CleanOrigin", tags=None)
    assert _pair_is_admissible("CleanOrigin", "InfoNote", all_data, info_data, notat_data)
    
    # Case 2: Origin has auto-links tag, Target is InfoNote -> Inadmissible
    all_data["AutoLinkedOrigin"] = mock_note_data("AutoLinkedOrigin", tags={'_auto/links_added'})
    assert not _pair_is_admissible("AutoLinkedOrigin", "InfoNote", all_data, info_data, notat_data)
    
    # Case 3: Origin has auto-links tag, Target is NotatNote -> Admissible 
    # (because relied_note 'NotatNote' is NOT in info_data, it's in notat_data)
    assert _pair_is_admissible("AutoLinkedOrigin", "NotatNote", all_data, info_data, notat_data)

# ---------------------------------------------------------
# Test 6: Similar Notation Logic (UPDATED for jarowinkler_similarity)
# ---------------------------------------------------------
def test_similar_notation_logic():
    n1 = mock_note_data("N1"); n1.parsed.notation_str = "Gal(L/K)"
    n2 = mock_note_data("N2"); n2.parsed.notation_str = "Gal(F/E)"
    n3 = mock_note_data("N3"); n3.parsed.notation_str = "Spec(R)"
    
    data = {"N1": n1, "N2": n2, "N3": n3}
    
    # Patch the function directly where it is imported in your main module
    # Assuming your main module is called "__main__" in the notebook context
    with patch('__main__.jarowinkler_similarity') as mock_similarity:
        # Define side effects: High similarity (>0.9) for N1-N2, Low for others
        def sim_side_effect(s1, s2):
            if "Gal" in s1 and "Gal" in s2: return 0.95
            return 0.1
        
        mock_similarity.side_effect = sim_side_effect
        
        sim_pairs = _similar_notation_pairs(data)
        
        # Should find N1->N2 and N2->N1
        assert "N2" in sim_pairs["N1"]
        assert "N1" in sim_pairs["N2"]
        assert "N3" not in sim_pairs

# Run all
test_link_types_extraction()
test_note_counting_logic()
test_positive_instance_extraction()
test_negative_instance_sampling()
test_pair_admissibility()
test_similar_notation_logic()